# ISIC 2024 `strict_input + iddx_full` Dataset Contract

이 노트북은 ISIC 2024 `train-metadata.csv`를 바탕으로, tabular/fusion model 학습에서 사용할 `strict_input`과 같은 sample의 `iddx_full`을 안전하게 추적하기 위한 데이터셋 계약 문서다.

현재 논문 방향의 기본 입력은 `image + strict_input`이다. `iddx_full` 및 그 파생값은 ordinary inference-time feature가 아니며, 후보 실험에서만 사용할 수 있는 train-only privileged supervision 신호다.

## 0. 개요

### 0.1 이 노트북의 범위

1. 데이터와 누수 위험 요약
2. 컬럼 role registry
3. `strict_input` 계약
4. `iddx_full` train-only 계약
5. Label Embedding / Graph Metric Learning 후보 전처리 계약
6. split / preprocessing leakage contract
7. handoff checklist

### 0.2 최종 규칙

- `strict_input`은 validation/test/inference에서 요구할 수 있는 ordinary metadata 입력이다.
- `iddx_full`은 같은 `isic_id` row에서 추적하되, train-only candidate branch에서만 사용한다.
- `iddx_label_group` 같은 `iddx_full` 파생값은 validation/test/inference scoring에 들어가지 않는다.
- split은 `patient_id -> lesion_id -> isic_id` 구조를 보존해야 한다.
- 데이터 분포에서 학습되는 preprocessing state는 fold-local train split에서만 fit한다.

### 0.3 이 노트북에서 하지 않는 것

이 파일은 EDA 전체나 모델링 노트북이 아니다. 다음 작업은 이 노트북의 active scope에서 제외한다.

- feature engineering 설계
- PCA/VIF 기반 변수 선택
- winsorization, log1p, robust scaling 같은 preprocessing artifact 저장
- tail enrichment 기반 변수 판단
- 모델 학습 또는 threshold 선택
- paper-valid final split 생성

필요하면 위 항목은 별도 legacy EDA 또는 실험 노트북에서 다룬다. 이 파일은 `strict_input + iddx_full` dataset contract와 leakage boundary를 고정하는 데 집중한다.

### 0.4 용어 통일

- `strict_input`: 추론 시점에 사용 가능한 ordinary tabular metadata 입력.
- `strict_input_numeric`: `strict_input` 안의 원시 수치형 metadata 컬럼.
- `strict_input_categorical`: `strict_input` 안의 원시 범주형 metadata 컬럼.
- `iddx_full_train_text`: 같은 `isic_id` sample에 연결되는 `iddx_full` 원문 또는 정규화 텍스트.
- `iddx_label_group`: train-only candidate에서 `iddx_full`을 수동 그룹화한 `MEL/BCC/SCC/NV/OTHER` label.
- `reference_only`: 일반 inference input으로 쓰지 않는 leakage-risk reference field 묶음.
- `inference contract`: 최종 예측 시 `iddx_full` 없이 `image + strict_input` 또는 `strict_input`만 요구해야 한다는 입력 계약.

## 1. 데이터와 위험 요약

이 장은 계약 문서에 필요한 최소 데이터 사실만 확인한다. 원자료는 읽기 전용으로 다루고, 이 노트북은 산출물을 저장하지 않는다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    (
        candidate
        for candidate in [Path.cwd(), *Path.cwd().parents]
        if (candidate / 'data/raw/isic_2024_challenge/train-metadata.csv').exists()
    ),
    Path('/home/ubuntu/wksp/paper_ajou_dev'),
)
TRAIN_PATH = PROJECT_ROOT / 'data/raw/isic_2024_challenge/train-metadata.csv'
PUBLIC_TEST_PATH = PROJECT_ROOT / 'data/raw/isic_2024_challenge/test-metadata.csv'
train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
public_test_df = pd.read_csv(PUBLIC_TEST_PATH, low_memory=False)
df = train_df.copy()
target_name_map = {0: 'benign', 1: 'malignant'}

summary_df = pd.DataFrame(
    [
        {'item': 'train_rows', 'value': len(train_df)},
        {'item': 'train_patients', 'value': train_df['patient_id'].nunique()},
        {'item': 'train_lesion_id_non_null', 'value': int(train_df['lesion_id'].notna().sum())},
        {'item': 'positive_rows', 'value': int(train_df['target'].sum())},
        {'item': 'positive_rate_pct', 'value': round(float(train_df['target'].mean() * 100), 5)},
        {'item': 'public_test_rows', 'value': len(public_test_df)},
    ]
)

patient_target_summary_df = train_df.groupby('patient_id').agg(
    n_rows=('isic_id', 'size'),
    positive_rows=('target', 'sum'),
    lesion_id_non_null=('lesion_id', lambda s: int(s.notna().sum())),
).reset_index()
patient_target_summary_df['has_malignant'] = patient_target_summary_df['positive_rows'] > 0
patient_risk_summary_df = pd.DataFrame(
    [
        {'item': 'patients_with_malignant', 'value': int(patient_target_summary_df['has_malignant'].sum())},
        {'item': 'max_rows_per_patient', 'value': int(patient_target_summary_df['n_rows'].max())},
        {'item': 'median_rows_per_patient', 'value': round(float(patient_target_summary_df['n_rows'].median()), 3)},
    ]
)

display(Markdown('**표 1-a. 데이터 규모와 target 희귀성**'))
display(summary_df)
display(Markdown('**표 1-b. patient-level grouping 위험 요약**'))
display(patient_risk_summary_df)

### 1.1 해석

이 데이터는 malignant target이 극도로 희귀하고, row가 독립 표본이 아니라 `patient_id` 단위 cluster 구조를 가진다. 따라서 paper-facing 실험에서 row-level random split은 허용하지 않는다.

public `test-metadata.csv`는 schema 확인 정도에만 사용하고, EDA/feature choice/model selection의 근거로 사용하지 않는다.

## 2. 컬럼 role registry

이 장은 각 컬럼이 어떤 regime에 속하는지 고정한다. 목적은 입력 후보를 늘리는 것이 아니라, inference-time input과 train-only/reference-only 신호를 분리하는 것이다.

In [ ]:
REGIME_DISPLAY_MAP = {
    'strict': 'strict_input',
    'relaxed': 'Relaxed context optional sensitivity check',
    'oracle': 'Train-only iddx_full text',
    'reference': 'Reference only / excluded reference fields',
    'not_feature': 'Identifier / grouping key',
    'label': 'Label',
}

RELAXED_EXTRA_COLS = ['attribution', 'copyright_license']
TRAIN_ONLY_IDDX_FULL_TEXT_COLS = ['iddx_full']
REFERENCE_ONLY_COLS = [
    'lesion_id', 'iddx_1', 'iddx_2', 'iddx_3', 'iddx_4', 'iddx_5',
    'mel_mitotic_index', 'mel_thick_mm', 'tbp_lv_dnn_lesion_confidence',
]
STRICT_COLS = [
    'age_approx', 'sex', 'anatom_site_general', 'clin_size_long_diam_mm',
    'image_type', 'tbp_tile_type', 'tbp_lv_A', 'tbp_lv_Aext', 'tbp_lv_B',
    'tbp_lv_Bext', 'tbp_lv_C', 'tbp_lv_Cext', 'tbp_lv_H', 'tbp_lv_Hext',
    'tbp_lv_L', 'tbp_lv_Lext', 'tbp_lv_areaMM2', 'tbp_lv_area_perim_ratio',
    'tbp_lv_color_std_mean', 'tbp_lv_deltaA', 'tbp_lv_deltaB', 'tbp_lv_deltaL',
    'tbp_lv_deltaLB', 'tbp_lv_deltaLBnorm', 'tbp_lv_eccentricity', 'tbp_lv_location',
    'tbp_lv_location_simple', 'tbp_lv_minorAxisMM', 'tbp_lv_nevi_confidence',
    'tbp_lv_norm_border', 'tbp_lv_norm_color', 'tbp_lv_perimeterMM',
    'tbp_lv_radial_color_std_max', 'tbp_lv_stdL', 'tbp_lv_stdLExt',
    'tbp_lv_symm_2axis', 'tbp_lv_symm_2axis_angle', 'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z',
]

CATEGORY_MAP = {
    'label': ['target'],
    'identifier_grouping': ['isic_id', 'patient_id'],
    'strict_input': STRICT_COLS,
    'relaxed_context_optional': RELAXED_EXTRA_COLS,
    'train_only_iddx_full_text': TRAIN_ONLY_IDDX_FULL_TEXT_COLS,
    'reference_only_excluded': REFERENCE_ONLY_COLS,
}


def assign_regime(column):
    if column == 'target':
        return 'label'
    if column in ['isic_id', 'patient_id']:
        return 'not_feature'
    if column in TRAIN_ONLY_IDDX_FULL_TEXT_COLS:
        return 'oracle'
    if column in REFERENCE_ONLY_COLS:
        return 'reference'
    if column in RELAXED_EXTRA_COLS:
        return 'relaxed'
    if column in STRICT_COLS:
        return 'strict'
    return 'unassigned'

registry_rows = []
for column in df.columns:
    registry_rows.append(
        {
            'column': column,
            'regime': assign_regime(column),
            'regime_display': REGIME_DISPLAY_MAP.get(assign_regime(column), 'Unassigned'),
            'dtype': str(df[column].dtype),
            'null_ratio_pct': round(float(df[column].isna().mean() * 100), 4),
            'n_unique': int(df[column].nunique(dropna=True)),
        }
    )
column_registry_df = pd.DataFrame(registry_rows)
regime_summary_df = column_registry_df['regime_display'].value_counts().rename_axis('regime').to_frame('column_count')
missing_registry_columns = sorted(set(STRICT_COLS + RELAXED_EXTRA_COLS + TRAIN_ONLY_IDDX_FULL_TEXT_COLS + REFERENCE_ONLY_COLS + ['isic_id', 'patient_id', 'target']) - set(df.columns))

if missing_registry_columns:
    raise KeyError(f'계약 컬럼이 train metadata에 없습니다: {missing_registry_columns}')

display(Markdown('**표 2-a. regime별 컬럼 수**'))
display(regime_summary_df)
display(Markdown('**표 2-b. role registry**'))
display(column_registry_df.sort_values(['regime', 'column']).reset_index(drop=True))

### 2.1 해석

`Relaxed context`는 active 입력 후보가 아니라 source/context sensitivity check다. `Reference only / excluded reference fields`는 ordinary input 후보가 아니라 accidental leakage를 막기 위한 제외 registry다.

`iddx_full`만 train-only privileged text candidate로 남긴다. `iddx_1~5`, `mel_*`, `tbp_lv_dnn_lesion_confidence`, `lesion_id`는 분석/감사용 reference로만 둔다.

## 3. `strict_input` 계약

`strict_input`은 validation/test/inference 시점에 요구할 수 있는 ordinary tabular metadata 입력이다. 이 장은 컬럼 목록과 최소 결측/자료형 profile만 고정한다.

In [ ]:
strict_input_profile_df = column_registry_df.loc[column_registry_df['column'].isin(STRICT_COLS)].copy()
strict_input_profile_df['input_contract'] = 'ordinary inference-time tabular input'
strict_input_numeric_cols = [column for column in STRICT_COLS if pd.api.types.is_numeric_dtype(df[column])]
strict_input_categorical_cols = [column for column in STRICT_COLS if column not in strict_input_numeric_cols]
strict_input_summary_df = pd.DataFrame(
    [
        {'item': 'strict_input_columns', 'value': len(STRICT_COLS)},
        {'item': 'numeric_columns', 'value': len(strict_input_numeric_cols)},
        {'item': 'categorical_columns', 'value': len(strict_input_categorical_cols)},
        {'item': 'max_null_ratio_pct', 'value': round(float(strict_input_profile_df['null_ratio_pct'].max()), 4)},
    ]
)

display(Markdown('**표 3-a. strict_input 요약**'))
display(strict_input_summary_df)
display(Markdown('**표 3-b. strict_input 컬럼 계약**'))
display(strict_input_profile_df[['column', 'dtype', 'null_ratio_pct', 'n_unique', 'input_contract']].reset_index(drop=True))

### 3.1 해석

`strict_input`은 원시 metadata 중심의 일반 입력 계약이다. 이 노트북에서는 새로운 tabular feature를 만들거나 변수 선택을 하지 않는다.

결측 대체, scaling, categorical encoding, missing indicator 같은 preprocessing state는 실제 실험의 fold-local train split에서만 fit해야 한다.

## 4. `iddx_full` train-only 계약

`iddx_full`은 ordinary tabular feature가 아니다. 같은 `isic_id` sample에 연결된 train-only text field로 추적하며, tokenizer/LM encoder 또는 auxiliary target 생성은 train split 안에서만 허용한다.

In [ ]:
required_sample_columns = ['isic_id', 'patient_id', 'lesion_id', 'target', *STRICT_COLS, 'iddx_full']
missing_sample_columns = [column for column in required_sample_columns if column not in df.columns]
if missing_sample_columns:
    raise KeyError(f'sample-level contract 컬럼이 없습니다: {missing_sample_columns}')

sample_alignment_check_df = pd.DataFrame(
    [
        {'check': 'isic_id_unique', 'pass': bool(df['isic_id'].is_unique), 'value': int(df['isic_id'].nunique())},
        {'check': 'patient_id_present', 'pass': bool(df['patient_id'].notna().all()), 'value': int(df['patient_id'].isna().sum())},
        {'check': 'target_binary', 'pass': set(df['target'].dropna().unique()).issubset({0, 1}), 'value': sorted(df['target'].dropna().unique().tolist())},
        {'check': 'iddx_full_present', 'pass': bool(df['iddx_full'].notna().all()), 'value': int(df['iddx_full'].isna().sum())},
    ]
)

phase_input_contract_df = pd.DataFrame(
    [
        {
            'phase': 'train',
            'strict_input': 'required',
            'iddx_full_or_derived': 'optional train-only auxiliary signal',
            'allowed_use': 'tokenizer/LM encoder, label embedding target, auxiliary loss',
        },
        {
            'phase': 'validation',
            'strict_input': 'required',
            'iddx_full_or_derived': 'not used for scoring or threshold selection',
            'allowed_use': 'leakage audit only',
        },
        {
            'phase': 'test',
            'strict_input': 'required',
            'iddx_full_or_derived': 'not allowed',
            'allowed_use': 'none',
        },
        {
            'phase': 'inference',
            'strict_input': 'required',
            'iddx_full_or_derived': 'not required',
            'allowed_use': 'none',
        },
    ]
)

display(Markdown('**표 4-a. sample alignment checks**'))
display(sample_alignment_check_df)
display(Markdown('**표 4-b. phase별 입력 계약**'))
display(phase_input_contract_df)

### 4.1 해석

`strict_input`과 `iddx_full`은 반드시 같은 `isic_id` row에서 온 값이어야 한다. 다만 같은 row에 존재한다는 사실과 모델 입력으로 사용한다는 사실은 다르다.

validation/test/inference dataloader는 `iddx_full`, `iddx_label_group`, `iddx_label_metric_group` 없이 동작해야 한다.

## 5. Label Embedding / Graph Metric Learning 후보 전처리 계약

이 후보는 `iddx_full`을 직접 inference feature로 쓰지 않고, train-only auxiliary label 또는 prototype/metric target으로 변환하는 방식이다.

표현 컬럼은 `MEL/BCC/SCC/NV/OTHER` 5개 그룹을 유지한다. metric learning active anchor group은 기본적으로 `MEL/BCC/SCC/NV` 4개로 제한한다.

In [ ]:
LABEL_EMBEDDING_GROUPS = ['MEL', 'BCC', 'SCC', 'NV', 'OTHER']
LABEL_METRIC_ACTIVE_GROUPS = ['MEL', 'BCC', 'SCC', 'NV']
OTHER_AUX_WEIGHT_DEFAULT = 0.05


def map_iddx_full_to_label_group(value):
    text = str(value)
    if 'Basal cell carcinoma' in text:
        return 'BCC'
    if 'Squamous cell carcinoma' in text:
        return 'SCC'
    if 'Melanoma' in text:
        return 'MEL'
    if text.startswith('Benign::Benign melanocytic proliferations::Nevus'):
        return 'NV'
    return 'OTHER'

iddx_label_profile_df = df[['isic_id', 'patient_id', 'lesion_id', 'target', 'iddx_full']].copy()
iddx_label_profile_df['iddx_label_group'] = iddx_label_profile_df['iddx_full'].map(map_iddx_full_to_label_group)
iddx_label_profile_df['iddx_label_metric_group'] = iddx_label_profile_df['iddx_label_group'].where(
    iddx_label_profile_df['iddx_label_group'].isin(LABEL_METRIC_ACTIVE_GROUPS),
    pd.NA,
)
iddx_label_profile_df['iddx_label_is_metric_anchor'] = iddx_label_profile_df['iddx_label_group'].isin(LABEL_METRIC_ACTIVE_GROUPS)

label_group_summary_df = (
    iddx_label_profile_df.groupby('iddx_label_group', dropna=False)
    .agg(
        rows=('isic_id', 'size'),
        positives=('target', 'sum'),
        patients=('patient_id', 'nunique'),
        unique_iddx_full=('iddx_full', 'nunique'),
        metric_anchor_rows=('iddx_label_is_metric_anchor', 'sum'),
    )
    .reindex(LABEL_EMBEDDING_GROUPS)
)
label_group_summary_df['row_pct'] = (label_group_summary_df['rows'] / len(iddx_label_profile_df) * 100).round(4)
label_group_summary_df['positive_pct_in_group'] = (
    label_group_summary_df['positives'] / label_group_summary_df['rows'].replace(0, np.nan) * 100
).round(4)

label_embedding_column_contract_df = pd.DataFrame(
    [
        {
            'column': 'iddx_label_group',
            'values': 'MEL/BCC/SCC/NV/OTHER',
            'role': '5-way train-only candidate label management unit',
            'fit_scope': 'fixed rule; created only inside candidate pipeline',
        },
        {
            'column': 'iddx_label_metric_group',
            'values': 'MEL/BCC/SCC/NV or NA',
            'role': 'active metric/prototype anchor group; OTHER masked',
            'fit_scope': 'train fold objectives only',
        },
        {
            'column': 'iddx_label_is_metric_anchor',
            'values': 'True for MEL/BCC/SCC/NV, False for OTHER',
            'role': 'supervised contrastive/triplet/prototype anchor mask',
            'fit_scope': 'fixed mask; batch construction fold-local train-only',
        },
        {
            'column': 'iddx_label_aux_weight',
            'values': 'class-balanced for active groups; OTHER default 0.0-0.1',
            'role': 'auxiliary CE/prototype loss weight',
            'fit_scope': 'computed from train fold only',
        },
    ]
)

mapping_unit_check_df = pd.DataFrame(
    [
        {'iddx_full_example': 'Malignant::Malignant adnexal epithelial proliferations - Follicular::Basal cell carcinoma', 'expected_group': 'BCC'},
        {'iddx_full_example': 'Malignant::Malignant epidermal proliferations::Squamous cell carcinoma in situ', 'expected_group': 'SCC'},
        {'iddx_full_example': 'Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma in situ::Melanoma in situ, associated with a nevus', 'expected_group': 'MEL'},
        {'iddx_full_example': 'Benign::Benign melanocytic proliferations::Nevus::Nevus, Atypical, Dysplastic, or Clark', 'expected_group': 'NV'},
        {'iddx_full_example': 'Benign', 'expected_group': 'OTHER'},
    ]
)
mapping_unit_check_df['mapped_group'] = mapping_unit_check_df['iddx_full_example'].map(map_iddx_full_to_label_group)
mapping_unit_check_df['pass'] = mapping_unit_check_df['mapped_group'] == mapping_unit_check_df['expected_group']

if not mapping_unit_check_df['pass'].all():
    raise AssertionError('iddx_full manual group mapping check failed')

display(Markdown('**표 5-a. iddx_full label group summary**'))
display(label_group_summary_df)
display(Markdown('**표 5-b. label embedding candidate column contract**'))
display(label_embedding_column_contract_df)
display(Markdown('**표 5-c. manual mapping unit check**'))
display(mapping_unit_check_df)

### 5.1 해석 및 후속 구현 방향

5개 그룹은 label embedding 후보의 관리 단위이고, metric learning의 active anchor group은 `MEL/BCC/SCC/NV` 4개로 제한한다.

`OTHER`는 background/null group이다. supervised contrastive anchor, triplet anchor, clustering centroid 학습 대상에서 제외한다. Dataset contract와 auxiliary CE ablation에서는 `OTHER`를 보존하되, 기본 loss weight는 `0.0~0.1`로 둔다.

후속 후보 구현 순서는 다음과 같이 고정한다.

1. manual group auxiliary CE
2. prototype alignment
3. supervised contrastive / metric learning
4. graph prior regularization

이 후보는 strict baseline을 대체하지 않으며, 반드시 같은 patient-level split에서 strict baseline과 비교한다.

## 6. split / preprocessing leakage contract

이 장은 실제 split 산출물을 만들지 않는다. 대신 paper-facing experiment가 반드시 만족해야 하는 split과 preprocessing 계약을 고정한다.

In [ ]:
leakage_contract_df = pd.DataFrame(
    [
        {
            'contract': 'patient_disjoint_split',
            'rule': 'same patient_id must not appear across train/validation/test',
            'applies_to': 'all paper-facing experiments',
        },
        {
            'contract': 'lesion_alignment',
            'rule': 'lesion_id/isic_id remain attached to the same patient_id row',
            'applies_to': 'dataset export and dataloader',
        },
        {
            'contract': 'strict_preprocessing_train_only',
            'rule': 'imputation, scaling, categorical encoding, missing indicators fit on train fold only',
            'applies_to': 'strict_input preprocessing',
        },
        {
            'contract': 'iddx_derived_train_only',
            'rule': 'tokenizer vocab, LM fine-tuning, class weights, prototypes, metric batches fit/use train fold only',
            'applies_to': 'iddx_full candidate branch',
        },
        {
            'contract': 'validation_threshold_only',
            'rule': 'threshold and calibration selected on validation, never on test',
            'applies_to': 'metrics and reporting',
        },
        {
            'contract': 'inference_no_iddx',
            'rule': 'predict/inference API must not require iddx_full or iddx_label_group',
            'applies_to': 'deployment-facing path',
        },
    ]
)

phase_leakage_checklist_df = pd.DataFrame(
    [
        {'check': 'iddx_full-derived value not used in validation/test scoring', 'required': True},
        {'check': 'iddx_label_group not required by inference Dataset item', 'required': True},
        {'check': 'fold-local class weight/prototype/metric batch only', 'required': True},
        {'check': 'patient_id overlap audited before trusting any score', 'required': True},
        {'check': 'test fold not used for model choice or threshold selection', 'required': True},
    ]
)

display(Markdown('**표 6-a. leakage contract**'))
display(leakage_contract_df)
display(Markdown('**표 6-b. candidate leakage checklist**'))
display(phase_leakage_checklist_df)

### 6.1 해석

`iddx_full`을 같은 sample에서 추적하는 것 자체는 데이터셋 계약상 필요하다. 그러나 그 값이나 파생값이 validation/test scoring 또는 inference API에 들어가면 strict inference protocol이 아니다.

`iddx_full` candidate 결과는 항상 candidate-only privileged supervision으로 표시한다.

## 7. handoff checklist

이 노트북은 Dataset class와 training pipeline 구현을 직접 수행하지 않는다. 다음 구현 단계에서 확인해야 할 계약을 넘긴다.

In [ ]:
experiment_contract_df = pd.DataFrame(
    [
        {
            'experiment_group': 'strict_tabular_baseline',
            'train_inputs': 'strict_input',
            'validation_test_inference_inputs': 'strict_input',
            'uses_iddx_full': False,
            'paper_role': 'ordinary strict baseline',
        },
        {
            'experiment_group': 'image_strict_multimodal_baseline',
            'train_inputs': 'image + strict_input',
            'validation_test_inference_inputs': 'image + strict_input',
            'uses_iddx_full': False,
            'paper_role': 'main multimodal baseline',
        },
        {
            'experiment_group': 'strict_input_iddx_full_candidate',
            'train_inputs': 'strict_input + train-only iddx_full-derived auxiliary signal',
            'validation_test_inference_inputs': 'strict_input only for scoring/inference',
            'uses_iddx_full': 'train_only',
            'paper_role': 'privileged supervision candidate, compare against strict baseline',
        },
    ]
)

implementation_checklist_df = pd.DataFrame(
    [
        {'item': 'Dataset item can return isic_id/patient_id/lesion_id/target/strict_input', 'required': True},
        {'item': 'train dataset may expose iddx_full_train_text or iddx_label_group for candidate branch', 'required': True},
        {'item': 'validation/test/inference dataloader works without iddx_full', 'required': True},
        {'item': 'model.forward/predict does not require iddx_full in inference mode', 'required': True},
        {'item': 'results record config path, split version, threshold source, inference_requires_iddx_full=False', 'required': True},
    ]
)

display(Markdown('**표 7-a. baseline / candidate experiment contract**'))
display(experiment_contract_df)
display(Markdown('**표 7-b. implementation handoff checklist**'))
display(implementation_checklist_df)

### 7.1 최종 정리

이 노트북의 최종 결론은 다음과 같다.

- 메인 실험 입력은 `image + strict_input`이다.
- `iddx_full`은 ordinary feature가 아니라 train-only candidate signal이다.
- `iddx_label_group`은 `MEL/BCC/SCC/NV/OTHER`로 관리하되, metric anchor는 `MEL/BCC/SCC/NV`만 기본 사용한다.
- `OTHER`는 background/null group으로 둔다.
- inference path는 `iddx_full` 또는 그 파생값을 요구하지 않는다.
- patient-level split과 fold-local preprocessing 없이는 paper-facing claim으로 쓰지 않는다.